# AI Travel Planner using LangGraph
### Major Project | Google Colab | Python + LangChain + LangGraph + Groq

**Project objective:** Build an AI travel-planning assistant that accepts destination, dates/duration, budget, number of travellers, interests, and travel style, then produces a personalized day-wise itinerary with an estimated cost breakdown.

**Workflow:** Validate preferences → identify attractions → generate itinerary → estimate budget → check budget → revise once if necessary → produce final travel plan.

**Important:** The generated prices and travel suggestions are estimates, not live bookings or guaranteed current prices. Verify opening hours, transport, entry fees, weather, and safety information before travelling.

## 1. Install dependencies
Run this cell first. If Colab asks you to restart the runtime after installation, restart and run the cells again from the top.

In [1]:
%pip -q install -U langgraph langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 13.6 MB/s eta 0:00:00


## 2. Imports and API key
Create a Groq API key in your Groq account and enter it when prompted. Do not paste a secret key into a shared notebook or submit it to your LMS.

In [2]:
import os
import json
import re
from getpass import getpass
from typing import TypedDict, List, Dict, Any

from IPython.display import display, Markdown
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

api_key = getpass("Enter your Groq API key (input is hidden): ").strip()
if not api_key:
    raise ValueError("A Groq API key is required.")
os.environ["GROQ_API_KEY"] = api_key

print("API key configured for this Colab runtime.")

Enter your Groq API key (input is hidden): ··········
API key configured for this Colab runtime.


## 3. Enter trip preferences
Change the example values or run the cell and answer the prompts.

In [19]:
destination = input("Destination (e.g., Goa): ").strip()
start_date = input("Start date (YYYY-MM-DD): ").strip()
end_date = input("End date (YYYY-MM-DD): ").strip()
travelers = int(input("Number of travellers: "))
budget = float(input("Total budget in INR (₹): "))
interests_text = input("Interests, comma-separated (e.g., beaches, food, culture): ").strip()
travel_style = input("Travel style (relaxed / balanced / adventurous / luxury / budget): ").strip()

if not destination:
    raise ValueError("Destination cannot be empty.")
if travelers < 1:
    raise ValueError("Number of travellers must be at least 1.")
if budget <= 0:
    raise ValueError("Budget must be greater than zero.")

interests = [item.strip() for item in interests_text.split(",") if item.strip()]
if not interests:
    interests = ["local attractions", "food", "culture"]

trip = {
    "destination": destination,
    "start_date": start_date,
    "end_date": end_date,
    "travelers": travelers,
    "budget": budget,
    "interests": interests,
    "travel_style": travel_style or "balanced",
}
trip

Destination (e.g., Goa): pondicherry 
Start date (YYYY-MM-DD): 2026-10-12
End date (YYYY-MM-DD): 2026-10-20
Number of travellers: 12
Total budget in INR (₹): 10000
Interests, comma-separated (e.g., beaches, food, culture): food,weather 
Travel style (relaxed / balanced / adventurous / luxury / budget): balanced


{'destination': 'pondicherry',
 'start_date': '2026-10-12',
 'end_date': '2026-10-20',
 'travelers': 12,
 'budget': 10000.0,
 'interests': ['food', 'weather'],
 'travel_style': 'balanced'}

## 4. Define LangGraph state
The state carries the user's preferences and the results produced by each node. This follows the `TravelState` idea from the assignment brief and extends it with fields for budget checks and revision.

In [20]:
class TravelState(TypedDict, total=False):
    destination: str
    start_date: str
    end_date: str
    travelers: int
    budget: float
    interests: List[str]
    travel_style: str
    duration_days: int
    attractions: List[str]
    itinerary: str
    estimated_cost: float
    cost_breakdown: Dict[str, Any]
    budget_status: str
    revision_count: int
    final_plan: str
    error: str

## 5. Configure the language model

In [21]:
# If this model is unavailable for your account, replace it with a currently supported
# model listed in your Groq Console.
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    max_retries=2,
)

# Quick connection check
test = llm.invoke("Reply with exactly: Travel Planner model connected.")
print(test.content)

Travel Planner model connected.


## 6. Helper functions and graph nodes

In [22]:
def safe_text(response) -> str:
    """Convert a model response to plain text."""
    content = response.content
    if isinstance(content, str):
        return content.strip()
    return str(content).strip()


def call_model(system_prompt: str, user_prompt: str) -> str:
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])
    return safe_text(response)


def validate_preferences(state: TravelState) -> TravelState:
    """Normalize input and estimate trip duration from ISO dates when possible."""
    from datetime import date
    updated = dict(state)
    updated["destination"] = str(state.get("destination", "")).strip()
    updated["interests"] = [
        str(x).strip() for x in state.get("interests", []) if str(x).strip()
    ]
    updated["travel_style"] = str(state.get("travel_style", "balanced")).strip() or "balanced"
    updated["revision_count"] = int(state.get("revision_count", 0))

    try:
        start = date.fromisoformat(state["start_date"])
        end = date.fromisoformat(state["end_date"])
        days = (end - start).days + 1
        if days < 1:
            raise ValueError("End date must be the same as or later than start date.")
        updated["duration_days"] = days
    except KeyError:
        updated["duration_days"] = 1
    except ValueError as exc:
        # Accept blank dates for a duration estimate, but report malformed dates clearly.
        if "End date must" in str(exc):
            raise
        if not state.get("start_date") or not state.get("end_date"):
            updated["duration_days"] = 1
        else:
            raise ValueError("Enter dates in YYYY-MM-DD format.") from exc

    if not updated["destination"]:
        raise ValueError("Destination cannot be empty.")
    if int(state.get("travelers", 0)) < 1:
        raise ValueError("Number of travellers must be at least 1.")
    if float(state.get("budget", 0)) <= 0:
        raise ValueError("Budget must be greater than zero.")
    if not updated["interests"]:
        updated["interests"] = ["local attractions", "food", "culture"]

    return updated


def extract_attractions(state: TravelState) -> TravelState:
    """Suggest destination activities aligned with interests and travel style."""
    prompt = f'''
Create a practical shortlist of 8–12 attractions, experiences, food areas, or activities
for this trip. Prioritize the traveller's interests. Do not claim live availability.
Return a numbered list only.

Destination: {state["destination"]}
Interests: {", ".join(state["interests"])}
Travel style: {state["travel_style"]}
Trip length: {state["duration_days"]} day(s)
Travellers: {state["travelers"]}
'''
    attractions = call_model(
        "You are a careful travel-planning assistant. Avoid inventing exact opening hours or guaranteed prices.",
        prompt,
    )
    return {"attractions": [line.strip() for line in attractions.splitlines() if line.strip()]}


def generate_itinerary(state: TravelState) -> TravelState:
    """Generate a day-by-day schedule with sensible pacing."""
    prompt = f'''
Create a personalized day-wise itinerary in Markdown for the trip below.

Destination: {state["destination"]}
Dates: {state.get("start_date") or "not provided"} to {state.get("end_date") or "not provided"}
Duration: {state["duration_days"]} day(s)
Travellers: {state["travelers"]}
Total budget for the whole group: ₹{state["budget"]:,.0f}
Interests: {", ".join(state["interests"])}
Travel style: {state["travel_style"]}

Suggested attractions/activities:
{chr(10).join(state.get("attractions", []))}

Requirements:
- Make one section for every day, with a short area/theme title.
- Use time slots, meal breaks, and realistic travel pacing.
- Explain briefly why activities match the interests.
- Include a few low-cost alternatives.
- Do not assume accommodation or intercity transport has already been paid for.
- Do not fabricate live prices, opening hours, or booking availability.
- End with practical planning notes.
'''
    itinerary = call_model(
        "You are an experienced trip planner. Be practical, clear, inclusive, and transparent about uncertainty.",
        prompt,
    )
    return {"itinerary": itinerary}


def estimate_budget(state: TravelState) -> TravelState:
    """Ask the model for an indicative cost estimate and parse its JSON response."""
    prompt = f'''
Estimate a realistic indicative trip cost in Indian rupees for the itinerary below.
Return ONLY valid JSON with this exact structure:
{{
  "currency": "INR",
  "transport": 0,
  "accommodation": 0,
  "food": 0,
  "activities_and_entry": 0,
  "local_transport": 0,
  "miscellaneous": 0,
  "total": 0,
  "assumptions": ["assumption 1", "assumption 2"]
}}

Estimate for the whole group, not per person. Use non-negative numbers.
Destination: {state["destination"]}
Duration: {state["duration_days"]} day(s)
Dates: {state.get("start_date") or "not provided"} to {state.get("end_date") or "not provided"}
Travellers: {state["travelers"]}
Budget limit: ₹{state["budget"]:,.0f}
Travel style: {state["travel_style"]}
Interests: {", ".join(state["interests"])}
Itinerary:
{state.get("itinerary", "")}

Be conservative. If exact prices are unknown, label the estimate as approximate in assumptions.
'''
    raw = call_model(
        "You are a travel budget analyst. Return valid JSON only, with no Markdown fences.",
        prompt,
    )
    try:
        match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in model response.")
        data = json.loads(match.group(0))
        categories = [
            "transport", "accommodation", "food", "activities_and_entry",
            "local_transport", "miscellaneous"
        ]
        breakdown = {key: max(0, float(data.get(key, 0) or 0)) for key in categories}
        total = float(data.get("total", 0) or sum(breakdown.values()))
        if total <= 0:
            total = sum(breakdown.values())
        breakdown["assumptions"] = data.get("assumptions", [])
        breakdown["currency"] = "INR"
        return {"cost_breakdown": breakdown, "estimated_cost": total}
    except Exception:
        # Fallback keeps the workflow running if the model returns malformed JSON.
        fallback_total = float(state["budget"])
        breakdown = {
            "transport": 0, "accommodation": 0, "food": 0,
            "activities_and_entry": 0, "local_transport": 0,
            "miscellaneous": 0, "assumptions": [
                "The model response could not be parsed.",
                "No reliable cost estimate is available; manually research prices."
            ],
            "currency": "INR",
        }
        return {
            "cost_breakdown": breakdown,
            "estimated_cost": fallback_total,
            "error": "Could not parse a reliable budget estimate. Treat the estimate as unavailable.",
        }


def check_budget(state: TravelState) -> TravelState:
    estimate = float(state.get("estimated_cost", 0))
    limit = float(state["budget"])
    status = "within budget" if estimate <= limit else "over budget"
    return {"budget_status": status}


def revise_itinerary(state: TravelState) -> TravelState:
    """Ask for a lower-cost itinerary once when the estimate exceeds the budget."""
    prompt = f'''
Revise this itinerary to better fit the total group budget of ₹{state["budget"]:,.0f}.
Keep the destination, trip duration, travellers, interests, and travel style in mind.
Prioritize free/low-cost attractions, affordable local food, public transport, and fewer paid activities.
Do not remove essential breaks or make the schedule exhausting.
Return the complete revised day-wise itinerary in Markdown.

Current itinerary:
{state.get("itinerary", "")}

Previous estimated cost: ₹{state.get("estimated_cost", 0):,.0f}
'''
    revised = call_model(
        "You are a budget-conscious travel planner. Be realistic and do not promise exact prices.",
        prompt,
    )
    return {"itinerary": revised, "revision_count": int(state.get("revision_count", 0)) + 1}


def prepare_final_plan(state: TravelState) -> TravelState:
    """Combine the itinerary and budget into the final response."""
    breakdown = state.get("cost_breakdown", {})
    rows = []
    for key, label in [
        ("transport", "Transport"),
        ("accommodation", "Accommodation"),
        ("food", "Food"),
        ("activities_and_entry", "Activities and entry fees"),
        ("local_transport", "Local transport"),
        ("miscellaneous", "Miscellaneous"),
    ]:
        if key in breakdown:
            rows.append(f"| {label} | ₹{float(breakdown.get(key, 0)):,.0f} |")
    assumptions = breakdown.get("assumptions", [])
    assumptions_text = "\n".join(f"- {item}" for item in assumptions) if assumptions else "- Prices are approximate and must be checked."
    status = state.get("budget_status", "not checked")
    final = f'''
# AI Travel Plan: {state["destination"]}

**Dates:** {state.get("start_date") or "Not provided"} to {state.get("end_date") or "Not provided"}
**Duration:** {state.get("duration_days", 1)} day(s)
**Travellers:** {state["travelers"]}
**Budget for the group:** ₹{float(state["budget"]):,.0f}
**Interests:** {", ".join(state.get("interests", []))}
**Travel style:** {state.get("travel_style", "balanced")}

## Day-wise itinerary
{state.get("itinerary", "No itinerary was generated.")}

## Estimated budget breakdown

| Category | Estimated cost |
|---|---:|
{chr(10).join(rows)}
| **Estimated total** | **₹{float(state.get("estimated_cost", 0)):,.0f}** |

**Budget status:** {status.title()}
**Budget remaining / overrun:** ₹{float(state["budget"]) - float(state.get("estimated_cost", 0)):,.0f}

### Assumptions and limitations
{assumptions_text}

- These are AI-generated estimates, not live quotes or reservations.
- Verify prices, opening hours, local transport, weather, and safety advice before booking.
- This project does not book hotels, tickets, or transport.
'''
    if state.get("error"):
        final += f"\n\n**Notice:** {state['error']}"
    return {"final_plan": final}

## 7. Build the LangGraph workflow
The graph checks the estimate. If the estimate is over budget, it revises the itinerary once and recalculates the budget.

In [23]:
def route_after_budget_check(state: TravelState) -> str:
    if state.get("budget_status") == "over budget" and int(state.get("revision_count", 0)) < 1:
        return "revise"
    return "finalize"


builder = StateGraph(TravelState)
builder.add_node("validate_preferences", validate_preferences)
builder.add_node("extract_attractions", extract_attractions)
builder.add_node("generate_itinerary", generate_itinerary)
builder.add_node("estimate_budget", estimate_budget)
builder.add_node("check_budget", check_budget)
builder.add_node("revise_itinerary", revise_itinerary)
builder.add_node("prepare_final_plan", prepare_final_plan)

builder.add_edge(START, "validate_preferences")
builder.add_edge("validate_preferences", "extract_attractions")
builder.add_edge("extract_attractions", "generate_itinerary")
builder.add_edge("generate_itinerary", "estimate_budget")
builder.add_edge("estimate_budget", "check_budget")
builder.add_conditional_edges(
    "check_budget",
    route_after_budget_check,
    {"revise": "revise_itinerary", "finalize": "prepare_final_plan"},
)
builder.add_edge("revise_itinerary", "estimate_budget")
builder.add_edge("prepare_final_plan", END)

travel_graph = builder.compile()
print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


## 8. Run the travel planner

In [25]:
result = travel_graph.invoke(trip)
display(Markdown(result["final_plan"]))


# AI Travel Plan: pondicherry

**Dates:** 2026-10-12 to 2026-10-20  
**Duration:** 9 day(s)  
**Travellers:** 12  
**Budget for the group:** ₹10,000  
**Interests:** food, weather  
**Travel style:** balanced

## Day-wise itinerary
# Pondicherry – 9‑Day Budget‑Friendly Group Itinerary  
**Dates:** 12 Oct 2026 – 20 Oct 2026  
**Travelers:** 12 people  
**Total budget:** **₹10,000** (≈ ₹833 / person) – about **₹92 / day** each  
**Interests:** Food, pleasant post‑monsoon weather  
**Travel style:** Balanced – relaxed mornings, cultural sights, tasty experiences, plenty of free time  

> **How the numbers work** – All costs are **indicative** (prices can vary).  
> • **Accommodation:** 8 nights in a shared dormitory/hostel (≈ ₹55 – ₹60 / night / person).  
> • **Food:** Most meals are prepared together in the guest‑house kitchen using groceries bought at local markets (≈ ₹130 – ₹150 / person for the whole trip). Simple street‑food meals are used for the remaining meals.  
> • **Transport:** Round‑trip bus/train to Pondicherry (≈ ₹170 – ₹190 / person) + local bus/auto passes (≈ ₹30 / person).  
> • **Activities / donations:** Mostly free; optional donations (≈ ₹20 – ₹40 / person).  

The sum stays comfortably **under ₹833 per person**.

---

## Day 1 – Arrival & Settling In  

| Time | Activity | Why it fits | Approx. Cost (per person) |
|------|----------|-------------|---------------------------|
| 07:00‑09:30 | **Travel to Pondicherry** – inter‑city bus (or train) from the nearest city | Early travel avoids heat; group ticket is cheap | **₹180 – ₹190** (round‑trip share) |
| 09:45‑10:30 | **Check‑in** – *Aurobindo Guest House* (mixed dormitory) | Central, clean, kitchen access for group cooking | **₹55** (night 1) |
| 10:45‑12:00 | **Orientation walk** around the neighbourhood; locate a grocery shop & tea stall | Sets the base, finds cheap snack spots | Free |
| 12:15‑13:00 | **Lunch** – *Thattu Kadai* street‑food (masala dosa) | Filling, cheap, authentic | **₹30** |
| 13:30‑15:30 | **Rest / unpack** – freshen up, hydrate | Important after travel | – |
| 16:00‑17:30 | **Group tea break** – local tea stall (chai & biscuits) | Low‑cost social moment | **₹15** |
| 18:30‑19:30 | **Dinner** – *Baker Street* (simple bakery sandwich) | Light early dinner, easy on the wallet | **₹35** |
| 20:00‑20:30 | **Evening briefing** – review itinerary, assign day‑lead | Keeps everyone on the same page | – |

**Day 1 total per person:** **≈ ₹315** (includes one‑way travel; the return leg is counted later).

---

## Day 2 – Sunrise Beach & French‑Quarter Food Walk  

| Time | Activity | Why it fits | Approx. Cost |
|------|----------|-------------|--------------|
| 06:30‑07:30 | **Sunrise walk** on **Promenade Beach** | Free, cool, great photo ops | Free |
| 07:45‑08:15 | **Breakfast** – *Idli* & coconut chutney (prepared in hostel kitchen) | Bulk‑cooked, cheap | **₹10** |
| 09:00‑11:30 | **Self‑guided French‑Quarter food crawl** (use free map) – stop at: <br>• *Le Café* (croissant) <br>• *Baker Street* (pastry) <br>• *Café Ole* (filter coffee) | Sample local bakeries; limit to **two** small items per person | **₹70** (two snacks) |
| 12:00‑13:00 | **Lunch** – *Budget thali* at *Matsya* (or a similar local canteen) | Hearty, group‑friendly | **₹70** |
| 13:30‑15:30 | **Free time** – explore boutiques or relax on the beach | Flexibility, no cost | – |
| 16:00‑16:30 | **Snack** – street‑side sundal (chickpea) | Protein boost, cheap | **₹20** |
| 19:00‑20:00 | **Dinner** – *Surguru* (vegetarian South‑Indian set) | Good value, large portions | **₹70** |
| 21:00‑21:30 | **Night stroll** on the promenade (optional) | Free, pleasant breeze | Free |

**Day 2 total per person:** **≈ ₹250**

---

## Day 3 – Auroville Exploration (Mostly Free)  

| Time | Activity | Why it fits | Approx. Cost |
|------|----------|-------------|--------------|
| 07:30‑08:00 | **Breakfast** – poha (hostel kitchen) | Quick, cheap | **₹10** |
| 08:30‑09:00 | **Bus to Auroville** (public bus, 10 km) | Group transport, low fare | **₹20** |
| 09:30‑11:00 | **Matrimandir visit** – timed entry (donation optional) | Iconic, free entry | **₹20** (donation) |
| 11:15‑12:00 | **Auroville Bakery** – sample a slice of the famous bread (free sample) | Food interest, no cost | Free |
| 12:30‑13:30 | **Packed lunch** – tiffin prepared the night before (rice, sambar, pickle) | Saves money, fun group activity | **₹20** (ingredients share) |
| 14:00‑16:00 | **Auroville Beach** – quiet, less crowded | Free beach time | Free |
| 16:30‑17:00 | **Return bus** to Pondicherry | Included in earlier fare | – |
| 18:30‑19:30 | **Dinner** – *Madhuram* (budget South‑Indian) | Simple, tasty | **₹70** |
| 20:00‑20:30 | **Group debrief** – share photos, plan tomorrow | Free | – |

**Day 3 total per person:** **≈ ₹130**

---

## Day 4 – Ashram, Market & Local Flavours  

| Time | Activity | Why it fits | Approx. Cost |
|------|----------|-------------|--------------|
| 07:30‑08:00 | **Breakfast** – upma (hostel kitchen) | Light start | **₹10** |
| 08:30‑10:30 | **Sri Aurobindo Ashram** – free guided meditation & garden stroll (donation optional) | Wellness, free | **₹

## Estimated budget breakdown

| Category | Estimated cost |
|---|---:|
| Transport | ₹2,160 |
| Accommodation | ₹4,800 |
| Food | ₹1,800 |
| Activities and entry fees | ₹360 |
| Local transport | ₹360 |
| Miscellaneous | ₹200 |
| **Estimated total** | **₹9,680** |

**Budget status:** Within Budget  
**Budget remaining / overrun:** ₹320

### Assumptions and limitations
- Intercity bus/train round‑trip cost estimated at ₹180 per person (group ticket rates).
- Local transport (city buses, shared autos) estimated at ₹30 per person for the whole stay.
- Budget dormitory/hostel accommodation assumed at ₹50 per night per person (8 nights).
- Food cost includes groceries for group cooking and occasional street‑food meals, estimated at ₹150 per person for the 9‑day trip.
- Activities and entry fees are mostly free; optional donations and a few paid entries estimated at ₹30 per person.
- Miscellaneous covers unforeseen small expenses (e.g., toiletries, extra snacks) estimated at ₹200 total (~5% of total cost).
- All costs are approximate and rounded to the nearest ten rupees; actual prices may vary.
- Costs are calculated for the entire group of 12 travelers, not per individual.

- These are AI-generated estimates, not live quotes or reservations.
- Verify prices, opening hours, local transport, weather, and safety advice before booking.
- This project does not book hotels, tickets, or transport.


## 9. Optional: inspect graph state
This helps explain the LangGraph workflow during your project demonstration or viva.

In [18]:
print("Destination:", result.get("destination"))
print("Duration (days):", result.get("duration_days"))
print("Attractions suggested:", len(result.get("attractions", [])))
print("Budget status:", result.get("budget_status"))
print("Revision count:", result.get("revision_count", 0))
print("State keys:", list(result.keys()))

Destination: goa
Duration (days): 9
Attractions suggested: 10
Budget status: within budget
Revision count: 0
State keys: ['destination', 'start_date', 'end_date', 'travelers', 'budget', 'interests', 'travel_style', 'duration_days', 'attractions', 'itinerary', 'estimated_cost', 'cost_breakdown', 'budget_status', 'revision_count', 'final_plan']
